# Lección 08 — Sistemas Multi-Agente con Claude

En este notebook vas a implementar los 3 patrones de flujo multi-agente:
1. **Secuencial** — agentes en cadena (A → B → C)
2. **Concurrente** — agentes en paralelo con `asyncio`
3. **Condicional** — bifurcación según el resultado de revisión

In [ ]:
%pip install anthropic python-dotenv -q

In [ ]:
import anthropic
import asyncio
import json
from dotenv import load_dotenv

load_dotenv()
client = anthropic.Anthropic()

def llamar_claude(system: str, mensaje: str, max_tokens: int = 800) -> str:
    """Helper para llamar a Claude de forma simple."""
    respuesta = client.messages.create(
        model="claude-opus-4-5",
        max_tokens=max_tokens,
        system=system,
        messages=[{"role": "user", "content": mensaje}]
    )
    return respuesta.content[0].text

print("Setup listo.")

## Patrón 1 — Flujo Secuencial

Tres agentes trabajan en cadena para producir un artículo de blog:
- **Investigador** → busca información y puntos clave
- **Redactor** → escribe el artículo
- **Editor** → revisa y mejora el texto final

El output de cada agente es el input del siguiente.

In [ ]:
def agente_investigador(tema: str) -> str:
    """Investiga un tema y devuelve los puntos clave."""
    return llamar_claude(
        system="""Sos un investigador. Tu trabajo es analizar un tema y devolver:
        1. Definición breve (1 párrafo)
        2. 5 puntos clave sobre el tema
        3. 2 ejemplos concretos
        Sé conciso y factual.""",
        mensaje=f"Investigá este tema: {tema}"
    )

def agente_redactor(investigacion: str, tema: str) -> str:
    """Escribe un artículo basado en la investigación."""
    return llamar_claude(
        system="""Sos un redactor de contenido para una comunidad de IA en español.
        Escribís artículos claros, amigables y con ejemplos prácticos.
        Estructura: título, introducción, desarrollo (3 secciones), conclusión.
        Tono: accesible para personas sin experiencia técnica.""",
        mensaje=f"Escribí un artículo sobre: {tema}\n\nUsá esta investigación como base:\n{investigacion}",
        max_tokens=1200
    )

def agente_editor(articulo: str) -> str:
    """Revisa y mejora el artículo."""
    return llamar_claude(
        system="""Sos un editor senior. Revisás artículos y los mejorás:
        - Corregís errores de estilo y claridad
        - Simplificás términos técnicos innecesarios
        - Mejorás el gancho del título si hace falta
        - Asegurás que el tono sea consistente
        Devolvé el artículo mejorado, no comentarios sobre él.""",
        mensaje=f"Editá y mejorá este artículo:\n\n{articulo}",
        max_tokens=1200
    )


def pipeline_contenido(tema: str) -> dict:
    """Pipeline secuencial: Investigar → Redactar → Editar."""
    print(f"Tema: {tema}")
    
    print("\n[1/3] Investigando...")
    investigacion = agente_investigador(tema)
    print(f"  ✓ Investigación completada ({len(investigacion)} chars)")
    
    print("[2/3] Redactando...")
    borrador = agente_redactor(investigacion, tema)
    print(f"  ✓ Borrador completado ({len(borrador)} chars)")
    
    print("[3/3] Editando...")
    articulo_final = agente_editor(borrador)
    print(f"  ✓ Artículo final completado ({len(articulo_final)} chars)")
    
    return {
        "investigacion": investigacion,
        "borrador": borrador,
        "articulo_final": articulo_final
    }


resultado_pipeline = pipeline_contenido("Qué es un agente de IA y por qué cambia todo")
print("\n=== ARTÍCULO FINAL ===")
print(resultado_pipeline["articulo_final"])

## Patrón 2 — Flujo Concurrente

Un mismo tema se analiza en paralelo por 3 agentes especializados simultáneamente.
Usamos `asyncio` para ejecutarlos al mismo tiempo y luego un orquestador combina los resultados.

Esto es mucho más rápido que hacerlo de forma secuencial.

In [ ]:
import anthropic
import asyncio

# Para flujo concurrente usamos el cliente async de Anthropic
client_async = anthropic.AsyncAnthropic()

async def agente_async(nombre: str, system: str, mensaje: str) -> tuple:
    """Versión async de un agente Claude."""
    respuesta = await client_async.messages.create(
        model="claude-opus-4-5",
        max_tokens=600,
        system=system,
        messages=[{"role": "user", "content": mensaje}]
    )
    return nombre, respuesta.content[0].text


async def analisis_concurrente(tema: str) -> dict:
    """Lanza 3 agentes en paralelo y combina sus resultados."""
    print(f"Tema: {tema}")
    print("Lanzando 3 agentes en paralelo...\n")
    
    # Definir los 3 agentes
    tareas = [
        agente_async(
            "Analista Técnico",
            "Sos un experto técnico. Explicás el tema desde el punto de vista de implementación y tecnología. Sé específico con herramientas y conceptos técnicos.",
            f"Analizá técnicamente: {tema}"
        ),
        agente_async(
            "Analista de Negocios",
            "Sos un consultor de negocios. Analizás el impacto en productividad, costos y oportunidades comerciales. Usá números y casos reales cuando sea posible.",
            f"Analizá el impacto de negocios de: {tema}"
        ),
        agente_async(
            "Analista para Principiantes",
            "Sos un educador. Explicás el tema en términos simples para personas sin experiencia técnica. Usá analogías del día a día y evitá la jerga.",
            f"Explicá de forma simple para principiantes: {tema}"
        )
    ]
    
    # Ejecutar todos en paralelo
    resultados_raw = await asyncio.gather(*tareas)
    resultados = dict(resultados_raw)
    
    for nombre, texto in resultados.items():
        print(f"=== {nombre} ===")
        print(texto[:300] + "..." if len(texto) > 300 else texto)
        print()
    
    return resultados


# Ejecutar el análisis concurrente
resultados_concurrentes = await analisis_concurrente("Claude Code y la programación asistida por IA")

In [ ]:
# Orquestador: combina los 3 análisis en una visión unificada
def orquestador_combinar(tema: str, analisis: dict) -> str:
    """Combina los análisis paralelos en un resumen unificado."""
    contexto = f"Tema: {tema}\n\n"
    for nombre, texto in analisis.items():
        contexto += f"=== {nombre} ===\n{texto}\n\n"
    
    return llamar_claude(
        system="""Sos un orquestador. Recibís análisis de múltiples expertos sobre el mismo tema
        y los sintetizás en una visión unificada de 3-4 párrafos que integra las perspectivas
        técnica, de negocios y para principiantes.""",
        mensaje=f"Sintetizá estos análisis en una visión unificada:\n\n{contexto}",
        max_tokens=600
    )

print("=== SÍNTESIS UNIFICADA ===")
sintesis = orquestador_combinar("Claude Code", resultados_concurrentes)
print(sintesis)

## Patrón 3 — Flujo Condicional

Un agente genera contenido, un revisor lo evalúa y según el resultado:
- **Aprobado** → va a publicación
- **Rechazado** → vuelve a corrección

Este patrón es ideal para flujos de trabajo con control de calidad.

In [ ]:
def agente_generador(tema: str) -> str:
    """Genera un post para redes sociales."""
    return llamar_claude(
        system="""Generás posts para LinkedIn sobre IA. El post debe:
        - Tener entre 100 y 300 palabras
        - Incluir un dato concreto o estadística
        - Terminar con una pregunta para la audiencia
        - Usar entre 3 y 5 emojis relevantes""",
        mensaje=f"Generá un post de LinkedIn sobre: {tema}"
    )

def agente_revisor(post: str) -> dict:
    """Evalúa si el post cumple con los criterios de calidad."""
    resultado = llamar_claude(
        system="""Sos un revisor de contenido. Evaluás si un post cumple estos criterios:
        1. Entre 100 y 300 palabras
        2. Incluye dato concreto o estadística
        3. Termina con pregunta
        4. Tiene 3-5 emojis
        
        Respondé SOLO en JSON con esta estructura:
        {"aprobado": true/false, "criterios_fallidos": [lista], "sugerencias": "texto"}""",
        mensaje=f"Evaluá este post:\n\n{post}"
    )
    texto = resultado.strip()
    if texto.startswith("```"):
        texto = texto.split("\n", 1)[1].rsplit("```", 1)[0]
    return json.loads(texto)

def agente_corrector(post: str, sugerencias: str) -> str:
    """Corrige el post según las sugerencias del revisor."""
    return llamar_claude(
        system="Corregís posts de LinkedIn según el feedback del editor. Mantenés la idea original pero aplicás las mejoras indicadas.",
        mensaje=f"Post original:\n{post}\n\nAplicá estas correcciones:\n{sugerencias}"
    )

def agente_publicador(post: str) -> str:
    """Simula la publicación del post aprobado."""
    print("\n[PUBLICADO EN LINKEDIN]")
    return f"✅ Post publicado exitosamente.\n\nContenido final:\n{post}"


def flujo_condicional(tema: str, max_iteraciones: int = 3) -> str:
    """Flujo completo con revisión y corrección iterativa."""
    print(f"Generando post sobre: {tema}")
    
    post = agente_generador(tema)
    print(f"\nBorrador generado ({len(post.split())} palabras)")
    
    for iteracion in range(1, max_iteraciones + 1):
        print(f"\n[Revisión #{iteracion}]")
        evaluacion = agente_revisor(post)
        
        if evaluacion["aprobado"]:
            print("✓ APROBADO → Publicando")
            return agente_publicador(post)
        else:
            print(f"✗ RECHAZADO → Criterios fallidos: {evaluacion['criterios_fallidos']}")
            print(f"  Sugerencias: {evaluacion['sugerencias']}")
            if iteracion < max_iteraciones:
                print("  Corrigiendo...")
                post = agente_corrector(post, evaluacion["sugerencias"])
    
    print("\n⚠ Máximo de iteraciones alcanzado. Publicando la mejor versión.")
    return agente_publicador(post)


resultado_final = flujo_condicional("Por qué los agentes de IA van a cambiar el trabajo creativo")
print(resultado_final)

## Resumen

| Patrón | Cuándo usarlo | Ventaja |
|---|---|---|
| Secuencial | Pasos que dependen entre sí | Simple, predecible, fácil de debuggear |
| Concurrente | Tareas independientes que pueden ir en paralelo | Mucho más rápido |
| Condicional | Necesitás control de calidad o lógica de negocio | Flexible, maneja errores automáticamente |

Estos patrones se pueden combinar: un flujo secuencial donde uno de los pasos es concurrente, o un flujo condicional que dispara un pipeline secuencial.

---
En la **Lección 09** vemos la metacognición — cómo hacer que los agentes evalúen y mejoren sus propias respuestas.